In [33]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

import numpy as np

### Configurações Iniciais

In [34]:
# Lista de produtos para pesquisa
PRODUTOS_ELETRONICOS = [
    "Caixa de Som JBL Flip 6",
    "Smart TV Samsung 50 polegadas Crystal UHD",
    # "Smartphone Xiaomi Redmi Note 15",
    # "Smartphone Samsung Galaxy S23 Ultra",
    # "Tablet Apple iPad Air M2",
    # "Notebook Gamer Acer Nitro 5",
    # "Monitor Gamer LG Ultragear 27",
    # "Mouse Sem Fio Logitech G305",
    # "Teclado Mecânico Razer BlackWidow",
    # "Headset Gamer HyperX Cloud II",
    # "Fone Bluetooth Sony WH-1000XM5",
    # "Caixa de som JBL Boombox 3",
    # "Apple Watch Series 9",
    # "Console PlayStation 5 Slim",
    # "Console Xbox Series X",
    # "Placa de Vídeo RTX 4060 NVIDIA",
    # "Memória RAM Kingston Fury 8GB DDR4",
    # "SSD Kingston NV2 1TB NVMe",
    # "Processador Intel Core i5-13400F",
    # "Webcam Logitech C920 Full HD",
    # "Roteador Wi-Fi 6 TP-Link Archer",
    # "Impressora Epson EcoTank L3250",
    # "Carregador Portátil Baseus 20000mAh",
    # "Cabo HDMI 2.1 8K Baseus",
    # "Microfone Condensador HyperX QuadCast",
    "Suporte para Monitor articulado F80N"
]

# Armazenamento dos resultados
lista_produtos = []


options = Options()

# Remove a flag "navigator.webdriver = true" que identifica o Selenium
options.add_argument("--disable-blink-features=AutomationControlled")

# Remove o banner "Chrome está sendo controlado por software automatizado"
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Simula um usuário real
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

navegador = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Oculta o webdriver via JavaScript logo após abrir o navegador
navegador.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Configuração do Navegador
navegador.maximize_window()



# URL de referência
url_kabum = "https://www.kabum.com.br"
url_amazon = "https://www.amazon.com.br"
url_mercado_livre = "https://www.mercadolivre.com.br"

### Web Scraping - Kabum

In [ ]:
navegador.get(url_kabum)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "inputBusca") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        time.sleep(2) 
        
        busca = navegador.find_element("xpath", "//*[@id='inputBusca']")
        
        busca.clear()
        time.sleep(1) 
        
        busca.send_keys(item_pesquisa)
        time.sleep(1.5) 
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, ".desktop\:my-8")
            )
        )
        
        time.sleep(3)

        nomes_elementos = navegador.find_elements("class name", "h-40")[:3]
        
        precos_elementos = navegador.find_elements(
            "xpath", 
            "//div[contains(@class, 'flex gap-4 items-center')]/span[2]"
        )[:3]

        links_elementos = navegador.find_elements(
            "css selector",
            "a.flex.flex-col.relative.gap-4"
        )[:3]

        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                
                if i < len(precos_elementos):
                    texto_preco = precos_elementos[i].text.replace("R$", "").strip()
                    valor = f"R$ {texto_preco}" if texto_preco else "Sem preço"
                else:
                    valor = "Sem preço"
                # -----------------------------------
                
                link = links_elementos[i].get_attribute("href") if i < len(links_elementos) else "Sem link"
                
                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "KaBuM"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")
        time.sleep(3)

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")

<>:26: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<>:26: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
C:\Users\ricar\AppData\Local\Temp\ipykernel_28444\1238355674.py:26: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
  (By.CSS_SELECTOR, ".desktop\:my-8")



>>> Pesquisando por: Caixa de Som JBL Flip 6...
Encontrado: Caixa de Som Bluetooth Portátil Charge 6 JBL - Pre... | Valor: 996,98
Encontrado: Caixa De Som Portátil JBL Flip 6, Bluetooth, 20W R... | Valor: 889,00
Encontrado: Caixa De Som Jbl Flip 6, Portátil, Com Bluetooth, ... | Valor: 1.559,50

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Encontrado: Samsung Smart TV 50" Crystal UHD 4K U8600F 2025, X... | Valor: 2.219,90
Encontrado: Smart TV 50" Samsung UHD 4K Crystal UHD U8600F UN5... | Valor: 2.768,00
Encontrado: Smart Tv Samsung 50" 4k Crystal Business Wi-fi HDM... | Valor: 2.921,07

>>> Pesquisando por: Suporte para Monitor articulado F80N...
Encontrado: Suporte de Mesa Articulado para Monitor de 17" a 3... | Valor: 109,99
Encontrado: Suporte de Mesa Articulado para Monitores de 17" a... | Valor: 169,99
Encontrado: Suporte de Mesa Articulado para Monitor de 17" a 3... | Valor: 149,99

>>> Pesquisa concluída. Indo para próxima etapa...


In [ ]:

navegador.get(url_mercado_livre)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "cb1-edit") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        time.sleep(2) 
        
        busca = navegador.find_element(By.XPATH, "//*[@id='cb1-edit']")
        
        busca.clear()
        time.sleep(1) 
        
        busca.send_keys(item_pesquisa)
        time.sleep(1.5) 
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "#cb1-edit")
            )
        )
        
        time.sleep(3) 

        print("Capturando nomes e links...")
        nomes_e_links = navegador.find_elements(By.CSS_SELECTOR, "a.poly-component__title")[:3]
        
        print("Capturando preços...")
        precos_elementos = navegador.find_elements(By.CLASS_NAME, "poly-component__price")[:3]

        for i in range(len(nomes_e_links)):
            try:
                nome = nomes_e_links[i].text
                link = nomes_e_links[i].get_attribute("href")
                
                try:
                    inteiro = precos_elementos[i].find_element(By.CLASS_NAME, "andes-money-amount__fraction").text
                    
                    try:
                        centavos = precos_elementos[i].find_element(By.CLASS_NAME, "andes-money-amount__cents").text
                        valor = f"R$ {inteiro},{centavos}"
                    except:
                        valor = f"R$ {inteiro},00"
                        
                except Exception as e:
                    valor = "Sem preço"

                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Mercado Livre"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")
        time.sleep(3)

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")


>>> Pesquisando por: Caixa de Som JBL Flip 6...
Capturando nomes e links...
Capturando preços...
Encontrado: Alto-falante portátil Jbl Flip 7 Red... | Valor: R$ 789,30
Encontrado: Caixa De Som Bluetooth 30w Prova D Água Flip 6 Jbl... | Valor: R$ 750,00
Encontrado: Caixa De Som Jbl Flip 6 Bluetooth Potência 30w Pre... | Valor: R$ 928,08

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Capturando nomes e links...
Capturando preços...
Encontrado: Smart Tv Samsung Led 50 Lh50befh4ggxzd Led Crystal... | Valor: R$ 2.999,90
Encontrado: Smart Tv U8600f Crystal Uhd 4k 50 2025 Preto Samsu... | Valor: R$ 2.799,90
Encontrado: Samsung Smart TV 50” Crystal UHD 4K + Soundbar HW-... | Valor: R$ 2.599,90

>>> Pesquisando por: Suporte para Monitor articulado F80N...
Capturando nomes e links...
Capturando preços...
Encontrado: Suporte Articulado de Mesa F80N com Pistão a Gás p... | Valor: R$ 300,00
Encontrado: Suporte North Bayou F-80 Para Tv/monitor De 17 A 3... | Valor: R$ 120,40
Enc

In [ ]:
navegador.get(url_amazon)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "twotabsearchtextbox") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        time.sleep(2)
        
        busca = navegador.find_element("xpath", "//*[@id='twotabsearchtextbox']")
        
        busca.clear()
        
        time.sleep(1)
        
        busca.send_keys(item_pesquisa)
        
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        time.sleep(1)

        nomes_elementos = navegador.find_elements("css selector", "h2.a-size-base-plus span")[:3]
        precos_elementos = navegador.find_elements("css selector", "span.a-price")[:3]
        links_elementos = navegador.find_elements("css selector", "a.a-link-normal.s-line-clamp-4")[:3]

        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                
                try:
                    inteiro = precos_elementos[i].find_element("css selector", ".a-price-whole").text
                    centavos = precos_elementos[i].find_element("css selector", ".a-price-fraction").text
                    inteiro = inteiro.replace(",", "").replace(".", "")
                    valor = f"R$ {inteiro},{centavos}"
                except:  
                    valor = "Sem preço"

                link = links_elementos[i].get_attribute("href")

                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Amazon"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")
        time.sleep(3)

if lista_produtos:
    df = pd.DataFrame(lista_produtos)

    print("\n" + "="*60)
    print("PRÉVIA DOS DADOS COLETADOS:")
    print(df.head())

    df.to_csv("resultados.csv", index=False, encoding='utf-8-sig', sep=';')

    print("\n" + "="*60)
    print(f"ARQUIVO 'resultados_amazon.csv' GERADO COM SUCESSO!")
    print("="*60)
else:
    print("Nenhum produto foi capturado.")

time.sleep(1)
navegador.quit()

# ISSO AQUI TU COLOCA EM UMA CÉLULA SEPARADA, APÓS A ANALISE DESCRITIVA ESTIVER PRONTA. COLOQUEI APENAS AQUI PARA TESTAR ENVIANDO O CSV
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import os

print(">>> Iniciando processo de envio de e-mail...")


nome_arquivo = "resultados.csv" # AQUI TU COLOCA O PDF QUE VAI GERAR DA ANALISE
email_remetente = ""
senha_remetente = "" 
email_destinatario = ""


msg = MIMEMultipart()
msg['From'] = email_remetente
msg['To'] = email_destinatario
msg['Subject'] = "Relatório de Produtos - Automação Web"


if os.path.exists(nome_arquivo):
    corpo_email = "Olá,\n\nA automação foi concluída. O arquivo CSV com os resultados está anexado."
    msg.attach(MIMEText(corpo_email, 'plain'))
    
    try:
        with open(nome_arquivo, "rb") as anexo:
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(anexo.read())
            encoders.encode_base64(part)
            part.add_header(
                'Content-Disposition',
                f'attachment; filename={nome_arquivo}'
            )
            msg.attach(part)
            print(f"Arquivo '{nome_arquivo}' anexado com sucesso!")
    except Exception as e:
        print(f"Erro ao tentar ler e anexar o arquivo: {e}")
else:
    corpo_email = "Olá,\n\nA rotina de e-mail foi executada, mas o arquivo CSV não foi encontrado na pasta."
    msg.attach(MIMEText(corpo_email, 'plain'))
    print("Aviso: O arquivo CSV não foi encontrado. O e-mail irá sem anexo.")

try:
    servidor = smtplib.SMTP('smtp.gmail.com', 587)
    servidor.starttls() 
    servidor.login(email_remetente, senha_remetente)
    
    servidor.sendmail(email_remetente, email_destinatario, msg.as_string())
    servidor.quit()
    
    print(">>> E-mail enviado com sucesso!")
    
except Exception as e:
    print(f"Erro na conexão com o servidor de e-mail: {e}")


>>> Pesquisando por: Caixa de Som JBL Flip 6...
Encontrado: JBL PartyBox On-the-Go 2 - Altifalante portátil Bl... | Valor: R$ 1819,00
Encontrado: JBL, Caixa de Som, Go 4, Bluetooth, Portátil, Aura... | Valor: Sem preço
Encontrado: JBL, Caixa de Som, Xtreme 4, Bluetooth, Portátil, ... | Valor: R$ 246,00

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Encontrado: Smart TV 32" LG HD 32LR600B Processador α5 Ger6 AI... | Valor: R$ 978,00
Encontrado: Samsung Smart TV 70" Crystal UHD 4K U8500F 2025, X... | Valor: Sem preço
Encontrado: Samsung Smart TV 43" Crystal UHD 4K U8600F 2025... | Valor: R$ 3798,99

>>> Pesquisando por: Suporte para Monitor articulado F80N...
Encontrado: ELG, F80N, Suporte Articulado de Mesa, Pistão a Gá... | Valor: R$ 156,00
Encontrado: Suporte articulado com 4 movimentos para TV de 10”... | Valor: Sem preço
Encontrado: Suporte Monitor Articulado, Adequado para Suportar... | Valor: Sem preço

PRÉVIA DOS DADOS COLETADOS:
                             

In [38]:

# ─────────────────────────────────────────────
# DADOS DE EXEMPLO  (substitua pelo seu CSV/Excel)
# ─────────────────────────────────────────────

df = pd.read_csv("resultados.csv", sep=";")

# Para carregar de arquivo, comente o bloco acima e use:
# df = pd.read_csv("precos.csv")
# df = pd.read_excel("precos.xlsx")

# Garante tipo numérico


SEP = "=" * 60


# 1. Substituir linhas vazias ou apenas com 'R$ ,' por NaN (nulo)
df['Preço'] = df['Preço'].replace(r'^\s*R\$\s*,*\s*$', np.nan, regex=True)

# 2. Remover 'R$', converter vírgula decimal para ponto e tirar espaços
df['Preço'] = df['Preço'].str.replace('R$', '', regex=False)
df['Preço'] = df['Preço'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df['Preço'] = df['Preço'].str.strip()

# 3. Tratar os valores nulos (NaN) preenchendo com 0, e converter para inteiro
df['Preço'] = df['Preço'].astype(float)
df['Preço'] = df['Preço'].fillna(0)

ValueError: could not convert string to float: 'Sem preço'

In [ ]:
df.head()

In [ ]:

# ─────────────────────────────────────────────
# 1. PREÇO MÉDIO POR PRODUTO E LOJA
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("1. PREÇO MÉDIO POR PRODUTO E LOJA")
print(SEP)
media_loja = (
    df.groupby(["Produto", "Loja"])["Preço"]
    .mean()
    .reset_index()
    .rename(columns={"Preço": "Preço Médio"})
)
print(media_loja.to_string(index=False))


In [ ]:
idx_min = df.groupby("Produto")["Preço"].idxmin()
print(idx_min)

In [ ]:
# ─────────────────────────────────────────────
# 2. SITE MAIS BARATO POR PRODUTO
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("2. SITE MAIS BARATO POR PRODUTO")
print(SEP)
idx_min = df.groupby("Produto")["Preço"].idxmin()
mais_barato = df.loc[idx_min, ["Produto", "Loja", "Preço", "Link"]].reset_index(drop=True)
mais_barato.columns = ["Produto", "Loja Mais Barata", "Menor Preço", "Link"]
print(mais_barato.to_string(index=False))


In [ ]:

# ─────────────────────────────────────────────
# 3. ESTATÍSTICAS POR PRODUTO
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("3. ESTATÍSTICAS POR PRODUTO")
print(SEP)
stats = df.groupby("Produto")["Preço"].agg(
    Menor_Preço="min",
    Maior_Preço="max",
    Preço_Médio="mean",
).reset_index()

stats["Variação_%"] = (
    (stats["Maior_Preço"] - stats["Menor_Preço"]) / stats["Menor_Preço"] * 100
).round(2)

print(stats.to_string(index=False))


# ─────────────────────────────────────────────
# 4. SITE MAIS VANTAJOSO PARA A COMPRA COMPLETA
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("4. SITE MAIS VANTAJOSO PARA A COMPRA COMPLETA")
print(SEP)

# Soma total por loja (apenas lojas que têm TODOS os produtos)
total_por_loja = df.groupby("Loja").agg(
    Qtd_Produtos=("Produto", "nunique"),
    Total=("Preço", "sum"),
).reset_index()

total_produtos = df["Produto"].nunique()
lojas_completas = total_por_loja[total_por_loja["Qtd_Produtos"] == total_produtos].copy()

if lojas_completas.empty:
    print("Nenhuma loja possui todos os produtos. Ranking parcial:")
    lojas_completas = total_por_loja.copy()

lojas_completas = lojas_completas.sort_values("Total")
print(lojas_completas.to_string(index=False))

melhor_loja = lojas_completas.iloc[0]["Loja"]
melhor_total = lojas_completas.iloc[0]["Total"]
print(f"\n✅  Melhor loja para compra completa: {melhor_loja}  (R$ {melhor_total:,.2f})")


# ─────────────────────────────────────────────
# 5. ECONOMIA ESCOLHENDO SEMPRE O MENOR PREÇO
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("5. ECONOMIA ESCOLHENDO SEMPRE O MENOR PREÇO")
print(SEP)

menor_preço_total = stats["Menor_Preço"].sum()
maior_preço_total = stats["Maior_Preço"].sum()
economia = maior_preço_total - menor_preço_total
economia_pct = economia / maior_preço_total * 100

print(f"  Menor preço total (sempre o mais barato): R$ {menor_preço_total:>10,.2f}")
print(f"  Maior preço total (sempre o mais caro)  : R$ {maior_preço_total:>10,.2f}")
print(f"  Economia potencial                       : R$ {economia:>10,.2f}  ({economia_pct:.1f}%)")

print(f"\n{SEP}")
print("Detalhe por produto:")
print(SEP)
detalhe = stats[["Produto", "Menor_Preço", "Maior_Preço"]].copy()
detalhe["Economia"] = detalhe["Maior_Preço"] - detalhe["Menor_Preço"]
detalhe["Economia_%"] = (detalhe["Economia"] / detalhe["Maior_Preço"] * 100).round(2)
print(detalhe.to_string(index=False))